# QRGuard Decision decision-2026.03-r05 — frozen performance report

This notebook displays the saved local Fusion/Decision evidence. It does not retrain, promote, push, or deploy any model.

## Phase 0 — Reproducible workspace and Drive mount

In [ ]:
# Mount Drive and unpack the exact source bundle.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

BUNDLE_ZIP = Path('/content/drive/MyDrive/QRGuard_ML_Colab.zip')
WORK = Path('/content/qrguard_ml')
if not BUNDLE_ZIP.is_file():
    raise FileNotFoundError(
        f'Upload QRGuard_ML_Colab.zip to {BUNDLE_ZIP} before Run all.'
    )
print('Bundle SHA-256:', hashlib.sha256(BUNDLE_ZIP.read_bytes()).hexdigest().upper())
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as archive:
    archive.extractall(WORK)
REPO = WORK / 'QRGuard_ML_Colab' / 'QRGuard'
assert (REPO / 'ml_training/requirements.txt').is_file(), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / 'backend'))
print('Training source:', REPO)


## Phase 1 — Display locked Decision performance evidence

In [ ]:
from IPython.display import Image as DisplayImage, Markdown, display
import pandas as pd

PERF = REPO / 'ml_training/decision_layer/performance/decision-2026.03-r05'
metrics = json.loads((PERF / 'metrics.json').read_text(encoding='utf-8'))
assert metrics['version'] == 'decision-2026.03-r05'
assert metrics['gates_passed'] is True
assert metrics['promotion_requested'] is False

display(Markdown((PERF / 'DECISION_LAYER_PERFORMANCE.md').read_text(encoding='utf-8')))
display(Markdown('## Locked metrics JSON'))
display(metrics)
display(Markdown('## Per-cell results'))
display(pd.read_csv(PERF / 'per_cell_metrics.csv'))

for name in ('tier_confusion_matrix.png', 'score_distribution.png', 'ablation.png'):
    display(Markdown(f'### {name}'))
    display(DisplayImage(filename=str(PERF / name)))

print('The saved training run did not self-promote its outputs.')
print('The repository later promoted r01+r05 locally; external deployment is pending.')
